# optimizer-init-params-list — worked example 3: List-materialize from model.parameters() vs a custom generator

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-init-params-list`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The `list(params)` call in optimizer `__init__` works identically whether the caller passes `model.parameters()` (a generator), a list comprehension, a tuple, or any other iterable. In all cases, calling `list(...)` consumes the iterable once and produces a standard Python list that can be iterated as many times as needed.

## Worked solution

**Step 1 — Accept any iterable.**
Our optimizer's `__init__` accepts any `params` iterable. We immediately convert it: `self.params = list(params)`.

**Step 2 — Test with three input styles.**
We pass (a) `model.parameters()` (generator), (b) a list comprehension, and (c) a tuple. In all three cases, `self.params` should be a Python list of the same tensors.

**Step 3 — Verify the type is always `list`.**
`isinstance(opt.params, list)` should be `True` regardless of what was passed in.

**Step 4 — Verify length and tensor identity.**
The list should contain exactly as many entries as the model has parameters, and each entry should be the same tensor object (not a copy).

In [ ]:
import torch as t
import torch.nn as nn

class FlexSGD:
    def __init__(self, params, lr):
        self.params = list(params)   # works for any iterable
        self.lr = lr

    @t.no_grad()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p.data -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None

# --- exercise it ---
t.manual_seed(3)
model = nn.Sequential(nn.Linear(3, 3), nn.Linear(3, 1))
expected_count = sum(1 for _ in model.parameters())

# Style A: generator
opt_a = FlexSGD(model.parameters(), lr=0.01)
# Style B: list comp
opt_b = FlexSGD([p for p in model.parameters()], lr=0.01)
# Style C: tuple
opt_c = FlexSGD(tuple(model.parameters()), lr=0.01)

for label, opt in [('A (gen)', opt_a), ('B (list)', opt_b), ('C (tuple)', opt_c)]:
    print(f'{label}: type={type(opt.params).__name__}, len={len(opt.params)}')
    assert isinstance(opt.params, list), 'params must always be a list'
    assert len(opt.params) == expected_count, f'Expected {expected_count}, got {len(opt.params)}'

print(f'Expected param count: {expected_count}')
print('All styles produce list of correct length.')